# Generative AI 008 — Output Parsers

Parsers are the one part of this module that runs **end to end offline**. A
parser turns text into a Python object, and a stub chat model supplies the text
— so every claim here is a real measurement.

| Part | What we check |
|---|---|
| A | what each parser adds to every prompt: **none / 21 / 831** chars |
| B | one reply, three parsers, three Python types |
| C | the 3×3 table — which parser catches which failure |
| D | the source's fourth parser does not import at all |

Needs `langchain-core` and `pydantic`. No API key anywhere.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from langchain_core.output_parsers import (StrOutputParser, JsonOutputParser,
                                           PydanticOutputParser)
from langchain_core.prompts import PromptTemplate
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from pydantic import BaseModel, Field

import langchain, langchain_core
print("langchain", langchain.__version__, "| langchain_core", langchain_core.__version__)


class Person(BaseModel):
    """A fictional person."""
    name: str = Field(description="Name of the person")
    age: int = Field(gt=18, description="Age of the person, over 18")
    city: str = Field(description="City the person belongs to")

## Part A — What each parser costs you

A parser has two halves: instructions it adds to the prompt, and the conversion
it runs on the reply. People remember the second and forget the first — which is
where the cost lives.

In [ ]:
jp = JsonOutputParser()
pp = PydanticOutputParser(pydantic_object=Person)

print("StrOutputParser      : none (it has no get_format_instructions)")
print("JsonOutputParser     :", len(jp.get_format_instructions()), "characters")
print("PydanticOutputParser :", len(pp.get_format_instructions()), "characters")
print()
print("JsonOutputParser, in full:")
print("   ", repr(jp.get_format_instructions()))

ratio = len(pp.get_format_instructions()) / len(jp.get_format_instructions())
print(f"\nratio: {ratio:.0f}x more text, on EVERY call")
assert len(jp.get_format_instructions()) == 21
assert len(pp.get_format_instructions()) == 831

In [ ]:
print(pp.get_format_instructions()[:420], "...")
# The schema, plus an example of a valid instance and an invalid one.

> **Version note.** Older tutorials show `JsonOutputParser` emitting a whole
> paragraph. Here it emits 21 characters. If your numbers differ from a
> tutorial's, check the version before assuming either is wrong.

## Part B — One reply, three parsers

In [ ]:
GOOD = '{"name": "Ayesha", "age": 27, "city": "Colombo"}'

def run(parser, text, instructions=""):
    tpl = PromptTemplate(
        template="Generate a fictional {place} person.\n{format_instructions}",
        input_variables=["place"],
        partial_variables={"format_instructions": instructions},
    )
    return (tpl | FakeListChatModel(responses=[text]) | parser).invoke({"place": "Sri Lankan"})

for name, parser in (("StrOutputParser", StrOutputParser()),
                     ("JsonOutputParser", JsonOutputParser()),
                     ("PydanticOutputParser", pp)):
    out = run(parser, GOOD)
    print(f"{name:<22} -> {type(out).__name__:<14} {out}")

In [ ]:
# StrOutputParser returns a TextAccessor, not a plain str. Do not panic:
out = run(StrOutputParser(), GOOD)
print("type      :", type(out).__name__)
print("mro       :", [c.__name__ for c in type(out).__mro__])
print("is a str  :", isinstance(out, str))
print("behaves   :", out.upper()[:20], "| len", len(out))

assert isinstance(out, str)      # it SUBCLASSES str - check this, not type()

## Part C — The 3×3 table

Three replies, three parsers. This is the whole lesson in one cell.

In [ ]:
RESPONSES = {
    "valid JSON, age 27":  '{"name": "Ayesha", "age": 27, "city": "Colombo"}',
    "valid JSON, age 12":  '{"name": "Kid", "age": 12, "city": "Colombo"}',
    "prose, not JSON":     "I think the person is Ayesha, she is 27 and lives in Colombo.",
}
PARSERS = {
    "StrOutputParser":      StrOutputParser(),
    "JsonOutputParser":     JsonOutputParser(),
    "PydanticOutputParser": PydanticOutputParser(pydantic_object=Person),
}

print(f"{'':<22}" + "".join(f"{n:<24}" for n in PARSERS))
grid = {}
for case, text in RESPONSES.items():
    row = []
    for pname, parser in PARSERS.items():
        try:
            row.append(type(run(parser, text)).__name__)
        except Exception as e:
            row.append(type(e).__name__)
    grid[case] = row
    print(f"{case:<22}" + "".join(f"{c:<24}" for c in row))

In [ ]:
# The middle row is the one that matters.
assert grid["valid JSON, age 12"][1] == "dict"                    # Json ACCEPTED it
assert grid["valid JSON, age 12"][2] == "OutputParserException"   # Pydantic REJECTED it

print("The schema says age must be over 18.")
print("JsonOutputParser handed a twelve-year-old straight through as a dict;")
print("it has no schema, so nothing in it knows about your rule.")
print()
print("If that dict was about to become a database row, the difference")
print("between those two cells is a caught error against a corrupt record.")
print()
print("THAT is what the 40x prompt cost from Part A buys. Not JSON - the")
print("rejection.")

In [ ]:
# Bottom row: asked for JSON, given prose. Neither JSON parser guesses.
assert grid["prose, not JSON"][1] == "OutputParserException"
assert grid["prose, not JSON"][2] == "OutputParserException"
print("Both raise rather than repair or fall back.")
print("A loud failure you can catch and retry beats a dict with the wrong keys")
print("flowing onward. LangChain's OutputFixingParser is the opt-in retry -")
print("it sends the bad response back to the model, so it needs a real key.")

## Part D — The fourth parser does not exist here

Most material on this topic teaches `StructuredOutputParser`, built from
`ResponseSchema` objects. Worth actually trying the import.

In [ ]:
import importlib, pkgutil

for path in ("langchain.output_parsers",
             "langchain_classic.output_parsers",
             "langchain_community.output_parsers"):
    try:
        importlib.import_module(path)
        print(f"{path:<40} imports OK")
    except Exception as e:
        print(f"{path:<40} {type(e).__name__}")

print()
print(f"langchain {langchain.__version__} submodules:",
      sorted(m.name for m in pkgutil.iter_modules(langchain.__path__)))

In [ ]:
from langchain_core import output_parsers
names = sorted(n for n in dir(output_parsers) if n.endswith("OutputParser"))
print(len(names), "parsers in langchain_core.output_parsers:")
for n in names:
    print("   ", n)

assert "StructuredOutputParser" not in names
print("\nStructuredOutputParser and ResponseSchema are not on this stack.")
print("They lived in langchain.output_parsers in the 0.x line and did not")
print("move into the 1.x main package.")
print()
print("No loss: their niche was enforcing field NAMES without checking")
print("values - exactly the gap the middle row of Part C showed. Pydantic")
print("does that job and validates too.")

## What to take away

- **A parser is two halves**: instructions into the prompt, conversion on the
  reply. The first half is where the cost is.
- **StrOutputParser** returns a `TextAccessor` — a `str` subclass. Check
  `isinstance`, not `type`.
- **JsonOutputParser**: a `dict` for **21 characters**. Guarantees valid JSON and
  nothing else — it **accepted** a record that broke the schema.
- **PydanticOutputParser**: a validated object for **831 characters**, about
  **40x** more. It **raised** on that same record.
- **Both JSON parsers raise on prose.** They do not repair and do not guess.
- **`StructuredOutputParser` does not import** on langchain 1.2.15.

## Exercises

1. Add a fourth reply: valid JSON with an extra key the schema does not mention.
   Which parsers care?
2. `JsonOutputParser` cost 21 characters here. Write your own format
   instructions that are stricter but still short, and see how far you can get
   toward Pydantic's guarantee without its bulk.
3. The 831 characters are per call. At 10,000 calls a day and $3 per million
   input tokens, what does `PydanticOutputParser` cost per year over
   `JsonOutputParser`? Is it worth it for your use case?
4. Look up `OutputFixingParser` and write the chain you would use. What is the
   worst case — how many model calls can one request cost?
5. `CommaSeparatedListOutputParser` is in the list from Part D. Use it, and work
   out why a list parser is more fragile than a JSON one.